# The double slit, one particle at a time

**Professor Nano Labs · Quantum foundations**

**Try it live in your browser:** [The double slit lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/double-slit/)
**Watch first:** [What 3 experiments really tell us about quantum superposition](https://www.youtube.com/watch?v=L4YajTag6Ko)
**Watch later:** [The 7 postulates of quantum mechanics](https://www.youtube.com/watch?v=1J3z6_-8CP0) · [Illustrating the difference between pure coherent and mixed states](https://www.youtube.com/watch?v=TZ4FPPm-1TM) · [Fourier vs Heisenberg uncertainty principle](https://www.youtube.com/watch?v=jfwoUOOw5ms)

A particle's wave reaches each point of the screen through both slits, and the two contributions add: $P(y)\propto|\psi_1+\psi_2|^2=|\psi_1|^2+|\psi_2|^2+2\,\mathrm{Re}\,\psi_1^*\psi_2$. This notebook

1. solves the time-dependent Schrödinger equation in two dimensions by the split-step Fourier method, for a wave packet meeting a wall with two slits;
2. compares the pattern on the screen with Huygens' wavelets;
3. draws particles one at a time with the Born rule;
4. checks the duality relation between which-path knowledge and fringe visibility.

It uses the same grid and settings as the lab, and checks every number quoted in the lab's "Predict first" answers. It needs NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# units: hbar = m = 1, lengths in grid steps; the wavelength is 8 steps
N, LAM = 256, 8
K0 = 2 * np.pi / LAM
G = dict(x0=60, sx=12, sy=28, wall=104, thick=4, screen=216, edge=24, V0=2.5, dt=0.5, tEnd=330)
LX = (G['screen'] - G['wall'] - G['thick']) / LAM          # screen distance in wavelengths
YMAX = (N / 2 - G['edge']) / LAM                            # visible half-height of the screen, in wavelengths
y_of_row = np.arange(N) - N / 2

def slits(d, a, open_='both'):
    out = []
    if open_ != 'bottom': out.append((d / 2, a))
    if open_ != 'top': out.append((-d / 2, a))
    return out
print(f'screen {LX} wavelengths beyond the wall; packet energy k0^2/2 = {K0**2 / 2:.3f}, wall V0 = {G["V0"]} ({G["V0"] / (K0**2 / 2):.1f} times higher)')

## 1 · The simulation

Each time step applies half the potential as a phase, the kinetic energy as a phase in momentum space, and the other half of the potential: $\psi\to e^{-iV\Delta t/2}\,\mathcal F^{-1}e^{-ik^2\Delta t/2}\mathcal F\,e^{-iV\Delta t/2}\psi$. A border of 24 steps absorbs whatever reaches the edges. The screen column adds up $|\psi|^2$ over time, and the run tracks how much probability is beyond the wall, including what the edges beyond it have absorbed.

In [ ]:
def simulate(d, a, open_='both', dt=G['dt'], V0=G['V0'], t_end=G['tEnd'], keep_frame_at=None):
    x = np.arange(N); X, Y = np.meshgrid(x, y_of_row)                         # rows are y, columns are x
    wall = (X >= G['wall']) & (X < G['wall'] + G['thick'])
    for yc, w in slits(d, a, open_): wall &= ~(np.abs(Y - yc) < w / 2)
    V = np.where(wall, V0, 0.0)
    ev = np.exp(-0.5j * V * dt)
    e = np.minimum(np.minimum(X, N - 1 - X), np.minimum(np.arange(N)[:, None] + 0 * X, N - 1 - np.arange(N)[:, None] + 0 * X))
    u = np.where(e < G['edge'], (G['edge'] - e) / G['edge'], 0.0)
    mask = np.exp(-0.06 * u * u)
    k = 2 * np.pi * np.fft.fftfreq(N)
    KX, KY = np.meshgrid(k, k)
    ek = np.exp(-0.5j * (KX**2 + KY**2) * dt)
    xr = X - G['x0']
    psi = np.exp(-xr**2 / (4 * G['sx']**2) - Y**2 / (4 * G['sy']**2) + 1j * K0 * xr)
    psi /= np.sqrt(np.sum(np.abs(psi)**2))
    screen = np.zeros(N); lost_left = lost_right = 0.0; frame = None
    beyond = X >= G['wall']
    for step in range(int(round(t_end / dt))):
        psi = ev * np.fft.ifft2(ek * np.fft.fft2(ev * psi))
        p = np.abs(psi)**2
        removed = p * (1 - mask**2)
        lost_left += removed[~beyond].sum(); lost_right += removed[beyond].sum()
        psi *= mask
        screen += np.abs(psi[:, G['screen']])**2 * dt
        if keep_frame_at is not None and frame is None and (step + 1) * dt >= keep_frame_at: frame = np.abs(psi)
    p = np.abs(psi)**2
    through = p[beyond].sum() + lost_right
    back = p[~beyond].sum() + lost_left
    return dict(screen=screen, through=through, back=back, frame=frame)

run = simulate(3 * LAM, LAM, keep_frame_at=180)
print(f'd = 3 lambda, a = lambda: through the slits {run["through"]:.4f}, bounced back {run["back"]:.4f}, sum {run["through"] + run["back"]:.6f}')
assert abs(run['through'] + run['back'] - 1) < 1e-6 and round(100 * run['through'], 1) == 24.0

plt.figure(figsize=(5.2, 5))
plt.imshow(run['frame'][G['edge']:N - G['edge'], G['edge']:N - G['edge']], origin='lower', cmap='Purples',
           extent=[G['edge'] / LAM, (N - G['edge']) / LAM, -YMAX, YMAX], vmax=0.3 * run['frame'].max())
plt.axvline(G['screen'] / LAM, color='tab:orange', ls='--'); plt.xlabel('x (wavelengths)'); plt.ylabel('y (wavelengths)')
plt.title('|psi| at t = 180, behind the slits'); plt.tight_layout(); plt.show()

**Why $V_0\Delta t$ must stay below $\pi$.** The wall acts only through the phase $e^{-iV_0\Delta t}$, and a phase cannot tell $V_0\Delta t$ from $V_0\Delta t-2\pi$. With $\Delta t=1$ and $V_0=6$, the phase is 6 radians, the same as $-0.28$: the "wall" behaves like a shallow dip, and most of the packet walks straight through it.

In [ ]:
bad = simulate(3 * LAM, LAM, dt=1.0, V0=6.0, t_end=100)
print(f'V0 dt = 6 (> 2 pi): {bad["through"]:.2f} of the packet is beyond the wall at t = 100, against {simulate(3 * LAM, LAM, t_end=100)["through"]:.2f} with V0 dt = 1.25')
assert bad['through'] > 0.6

# and the time step itself: dt = 0.5 against dt = 0.25 (same wall)
fine = simulate(3 * LAM, LAM, dt=0.25)
rows = slice(G['edge'], N - G['edge'])
c = np.corrcoef(run['screen'][rows], fine['screen'][rows])[0, 1]
print(f'screen pattern, dt = 0.5 against dt = 0.25: correlation {c:.4f}')
assert c > 0.98

## 2 · Huygens' wavelets

Each point of an open slit sends out a two-dimensional wavelet $e^{ikr}/\sqrt r$, with the obliquity factor $(1+\cos\theta)/2$ and the packet's amplitude $e^{-y'^2/4\sigma_y^2}$ there. The intensities are averaged over the packet's spread of wavenumbers, $\sigma_k=1/2\sigma_x$, with five-point Gauss–Hermite quadrature, exactly as in the lab.

In [ ]:
GH_X, GH_W = np.polynomial.hermite.hermgauss(5)            # nodes and weights for exp(-z^2)
YS = np.linspace(-YMAX * LAM, YMAX * LAM, 833)               # the lab's screen rows, in steps
YL = YS / LAM

def slit_amp(yc, a, y, k, L):
    M = max(12, int(np.ceil(a * 3)))
    yp = yc - a / 2 + a * (np.arange(M) + 0.5) / M
    dy = y[:, None] - yp[None, :]; r = np.sqrt(L * L + dy * dy)
    w = np.exp(-yp**2 / (4 * G['sy']**2)) * 0.5 * (1 + L / r) / np.sqrt(r) * (a / M)
    return np.sum(w * np.exp(1j * k * r), axis=1)

def huygens(d, a, open_='both', y=YS):
    L = G['screen'] - G['wall'] - G['thick']; sk = 1 / (2 * G['sx'])
    S = slits(d, a, open_)
    I = np.zeros(len(y)); I1 = np.zeros(len(y)); I2 = np.zeros(len(y)); Xt = np.zeros(len(y))
    for z, w in zip(GH_X, GH_W):
        k = K0 + np.sqrt(2) * sk * z
        A = [slit_amp(yc, aa, y, k, L) for yc, aa in S] + [np.zeros(len(y))] * (2 - len(S))
        a1, a2 = np.abs(A[0])**2, np.abs(A[1])**2; x = 2 * np.real(np.conj(A[0]) * A[1])
        I1 += w * a1; I2 += w * a2; Xt += w * x; I += w * (a1 + a2 + x)
    s = GH_W.sum()
    return dict(I=I / s, I1=I1 / s, I2=I2 / s, X=Xt / s)

def peaks(I, y=YL):
    i = np.where((I[1:-1] > I[:-2]) & (I[1:-1] >= I[2:]) & (I[1:-1] > 0.04 * I.max()))[0] + 1
    return y[i]

H = huygens(3 * LAM, LAM)
yrows = y_of_row[rows] / LAM
sim = run['screen'][rows]
pred = huygens(3 * LAM, LAM, y=y_of_row[rows])['I']
c = np.corrcoef(sim, pred)[0, 1]
print(f'simulation against Huygens, d = 3 lambda: correlation {c:.3f}')
print('bright stripes (wavelengths), Huygens:   ', np.round(peaks(H['I']), 3))
print('bright stripes (wavelengths), simulation:', np.round(peaks(sim, yrows), 3))
assert c > 0.96

plt.figure(figsize=(6.5, 3.4))
plt.plot(yrows, sim / sim.max(), color='tab:orange', label='simulation')
plt.plot(YL, H['I'] / H['I'].max(), 'k--', label='Huygens')
plt.xlabel('y on the screen (wavelengths)'); plt.ylabel('brightness'); plt.legend(); plt.tight_layout(); plt.show()

## 3 · The questions on the wave (Questions 1 to 3)

In [ ]:
# Question 1: the stripes crowd together when the slits move apart
p3, p6 = peaks(huygens(3 * LAM, LAM)['I']), peaks(huygens(6 * LAM, LAM)['I'])
s3, s6 = p3[p3 > 0.05].min(), p6[p6 > 0.05].min()
print(f'first bright stripe: d = 3 lambda at {s3:.3f}, d = 6 lambda at {s6:.3f} (rule of thumb lambda L/d: {LX / 3:.2f}, {LX / 6:.3f})')
assert round(s3, 1) == 4.6 and round(s6, 1) == 2.3 and LX == 13.5 and round(LX / 3, 2) == 4.5 and round(LX / 6, 2) == 2.25
sim6 = simulate(6 * LAM, LAM)
ps3, ps6 = peaks(run['screen'][rows], yrows), peaks(sim6['screen'][rows], yrows)
f3, f6 = ps3[ps3 > 0.05].min(), ps6[ps6 > 0.05].min()
print(f'simulation: first side stripe at {f3:.3f} (d = 3 lambda) and {f6:.3f} (d = 6 lambda)')
assert abs(f3 - 5.1) < 0.07 and abs(f6 - 2.3) < 0.07   # 5.125 and 2.25 on the grid of rows

# Question 2: one slit, one hump that never goes dark
Ht = huygens(3 * LAM, LAM, 'top')
print(f'top slit only: peaks at {peaks(Ht["I"])}, darkest point {Ht["I"].min() / Ht["I"].max():.3f} of the brightest')
assert list(peaks(Ht['I'])) == [1.5] and Ht['I'].min() / Ht['I'].max() > 0.05

# Question 3: wider slits, three stripes, and more of the wave gets through
Hw = huygens(3 * LAM, 2 * LAM)
pw = peaks(Hw['I']); wide = simulate(3 * LAM, 2 * LAM)
print(f'a = 2 lambda: bright stripes at {np.round(pw, 3)}; through the slits {wide["through"]:.3f} (a = lambda: {run["through"]:.3f})')
assert len(pw) == 3 and round(pw.max(), 1) == 4.0 and round(100 * wide['through'], 1) == 41.5 and round(100 * run['through'], 1) == 24.0

## 4 · One particle at a time (Questions 4 to 6)

Each detection is drawn from $P(y)\propto|\psi|^2$ by inverting its cumulative distribution. The central stripe runs between the first minima either side of the middle.

In [ ]:
I = H['I']
c0 = np.argmin(np.abs(YL)); lo = c0; hi = c0
while lo > 1 and I[lo - 1] <= I[lo]: lo -= 1
while hi < len(I) - 2 and I[hi + 1] <= I[hi]: hi += 1
share = I[lo:hi + 1].sum() / I.sum()
print(f'central stripe from {YL[lo]:.2f} to {YL[hi]:.2f} wavelengths holds {share:.4f} of the probability on the screen')
assert round(YL[hi], 1) == 2.3 and round(100 * share) == 34
mean, sd = 10000 * share, np.sqrt(10000 * share * (1 - share))
print(f'of 10 000 particles, about {mean:.0f} +- {sd:.0f} land there')
assert round(mean, -1) == 3440 and round(sd, -1) == 50

rng = np.random.default_rng(1)
cdf = np.cumsum(I); cdf /= cdf[-1]
def draw(n): return np.interp(rng.random(n), cdf, YL)
BIN = 0.5; edges = np.arange(-YMAX, YMAX + 1e-9, BIN)
expected = np.histogram(YL, bins=edges, weights=I)[0] / I.sum()
def noise(y):
    h = np.histogram(y, bins=edges)[0]; e = len(y) * expected
    return np.sqrt(np.sum((h - e)**2) / np.sum(e**2))
ratios = [noise(draw(100)) / noise(draw(10000)) for _ in range(200)]
print(f'noise for 100 particles / noise for 10 000: {np.mean(ratios):.2f} on average over 200 runs (sqrt(100) = 10)')
assert 9 < np.mean(ratios) < 11

ten = draw(10)
print('ten particles land at', np.round(ten, 1), '(no pattern to see)')
fig, ax = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
for a_, n in zip(ax, [10, 300, 10000]):
    y = draw(n); a_.hist(y, bins=edges, color='tab:purple', alpha=0.6); a_.plot(edges[:-1] + BIN / 2, n * expected, 'k--'); a_.set_title(f'{n} particles'); a_.set_xlabel('y (wavelengths)')
plt.tight_layout(); plt.show()

## 5 · Which way did it go? (Questions 7 and 8)

With marker states $|\chi_1\rangle,|\chi_2\rangle$, the cross term is multiplied by $\langle\chi_1|\chi_2\rangle$. For a pure marker, $V=|\langle\chi_1|\chi_2\rangle|$ and $D=\sqrt{1-|\langle\chi_1|\chi_2\rangle|^2}$, so $D^2+V^2=1$.

In [ ]:
def contrast(H, D):
    c = np.argmin(np.abs(YL))
    return np.sqrt(max(0.0, 1 - D * D)) * H['X'][c] / (H['I1'][c] + H['I2'][c])

for D_ in [0.0, 0.6, 1.0]:
    print(f'D = {D_}: visibility {contrast(H, D_):.4f}, D^2 + V^2 = {D_**2 + contrast(H, D_)**2:.4f}')
assert abs(contrast(H, 0) - 1) < 1e-9 and round(contrast(H, 0.6), 3) == 0.800 and abs(contrast(H, 1)) < 1e-9

# the same visibility measured as (Imax - Imin)/(Imax + Imin) from a monochromatic, evenly lit pair of slits at the centre
psi1, psi2 = np.exp(1j * 0.0), np.exp(1j * np.pi)        # in step (bright) and out of step (dark), equal amplitudes
for D_ in [0.0, 0.6]:
    V = np.sqrt(1 - D_**2)
    Imax, Imin = 2 + 2 * V, 2 - 2 * V
    print(f'D = {D_}: (Imax - Imin)/(Imax + Imin) = {(Imax - Imin) / (Imax + Imin):.3f}')

plt.figure(figsize=(6.5, 3.4))
for D_, col in [(0, 'k'), (0.6, 'tab:purple'), (1, 'tab:red')]:
    plt.plot(YL, (H['I1'] + H['I2'] + np.sqrt(1 - D_**2) * H['X']) / H['I'].max(), color=col, label=f'D = {D_}')
plt.xlabel('y on the screen (wavelengths)'); plt.ylabel('brightness'); plt.legend(); plt.tight_layout(); plt.show()

## The wavelengths of real particles

The de Broglie wavelength $\lambda=h/p=h/\sqrt{2mE}$: for electrons, $\lambda\approx1.226\,\mathrm{nm}/\sqrt{E/\mathrm{eV}}$.

In [ ]:
h, me, e = 6.62607015e-34, 9.1093837e-31, 1.602176634e-19
lam = lambda E_eV: h / np.sqrt(2 * me * E_eV * e) * 1e9
print(f'electron at 50 eV: {lam(50):.3f} nm; at 1 eV: {lam(1):.3f} nm')
assert round(lam(50), 2) == 0.17 and round(lam(1), 1) == 1.2